In [6]:
import os
import numpy as np
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GRU,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Dataset Path
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"


# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load Data
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)
        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"
    y = np.loadtxt(y_path).astype(int)

    # 1~6 -> 0~5
    y = y - 1

    return X, y


def load_subjects():

    subject_path = (
        f"{DATA_PATH}/train/subject_train.txt"
    )

    subjects = np.loadtxt(
        subject_path
    ).astype(int)

    return subjects


# ============================================================
# 5. Load Dataset
# ============================================================

X_train, y_train = load_data("train")
X_test, y_test = load_data("test")

subjects_train = load_subjects()


print("\nDataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("subjects:", subjects_train.shape)


# ============================================================
# 6. Model
# ============================================================

def build_model():

    inputs = Input(
        shape=(128, 9)
    )

    # CNN Block 1
    x = Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    )(inputs)

    x = BatchNormalization()(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)

    # CNN Block 2
    x = Conv1D(
        filters=128,
        kernel_size=3,
        activation="relu"
    )(x)

    x = BatchNormalization()(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)

    # GRU
    x = GRU(
        64,
        dropout=0.2
    )(x)

    # Dense
    x = Dense(
        128,
        activation="relu"
    )(x)

    x = Dropout(
        0.3
    )(x)

    # Output
    outputs = Dense(
        6,
        activation="softmax"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ============================================================
# 7. Stratified Group K-Fold
# ============================================================

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)


# ============================================================
# 8. Result Storage
# ============================================================

fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []


# ============================================================
# 9. Cross Validation
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X_train,
        y_train,
        groups=subjects_train
    ),
    start=1
):

    print("\n")
    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)

    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    X_tr = X_train[train_idx]
    y_tr = y_train[train_idx]

    X_val = X_train[val_idx]
    y_val = y_train[val_idx]

    subjects_tr = subjects_train[train_idx]
    subjects_val = subjects_train[val_idx]

    # --------------------------------------------------------
    # Subject Leakage Check
    # --------------------------------------------------------

    overlap = np.intersect1d(
        np.unique(subjects_tr),
        np.unique(subjects_val)
    )

    print("Train shape       :", X_tr.shape)
    print("Validation shape  :", X_val.shape)

    print(
        "Train subjects    :",
        np.unique(subjects_tr)
    )

    print(
        "Validation subjects:",
        np.unique(subjects_val)
    )

    print(
        "Subject overlap   :",
        overlap
    )

    if len(overlap) == 0:
        print("No subject leakage.")
    else:
        print("WARNING: SUBJECT LEAKAGE!")

    # --------------------------------------------------------
    # Fold-specific Normalization
    # --------------------------------------------------------

    mean = X_tr.mean(
        axis=(0, 1),
        keepdims=True
    )

    std = X_tr.std(
        axis=(0, 1),
        keepdims=True
    )

    X_tr = (
        X_tr - mean
    ) / (std + 1e-8)

    X_val = (
        X_val - mean
    ) / (std + 1e-8)

    # --------------------------------------------------------
    # One-hot Encoding
    # --------------------------------------------------------

    y_tr_onehot = to_categorical(
        y_tr,
        num_classes=6
    )

    y_val_onehot = to_categorical(
        y_val,
        num_classes=6
    )

    # --------------------------------------------------------
    # Clear Session
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    # --------------------------------------------------------
    # Build Model
    # --------------------------------------------------------

    model = build_model()

    # --------------------------------------------------------
    # Early Stopping
    # --------------------------------------------------------

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(
        X_tr,
        y_tr_onehot,

        validation_data=(
            X_val,
            y_val_onehot
        ),

        epochs=50,

        batch_size=64,

        shuffle=True,

        callbacks=[
            early_stopping
        ],

        verbose=1
    )

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    y_prob = model.predict(
        X_val,
        verbose=0
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)

    # --------------------------------------------------------
    # Fold Results
    # --------------------------------------------------------

    print("\n")
    print("-" * 45)
    print(f"FOLD {fold} RESULTS")
    print("-" * 45)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")


# ============================================================
# 10. Stratified Group K-Fold Results
# ============================================================

print("\n")
print("=" * 60)
print("STRATIFIED GROUP K-FOLD RESULTS")
print("=" * 60)

print(
    f"Accuracy  : "
    f"{np.mean(fold_accuracies):.4f} "
    f"± {np.std(fold_accuracies):.4f}"
)

print(
    f"Precision : "
    f"{np.mean(fold_precisions):.4f} "
    f"± {np.std(fold_precisions):.4f}"
)

print(
    f"Recall    : "
    f"{np.mean(fold_recalls):.4f} "
    f"± {np.std(fold_recalls):.4f}"
)

print(
    f"F1-score  : "
    f"{np.mean(fold_f1s):.4f} "
    f"± {np.std(fold_f1s):.4f}"
)


# ============================================================
# 11. Individual Fold Results
# ============================================================

print("\n")
print("=" * 60)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 60)

for i in range(5):

    print(
        f"Fold {i+1}: "
        f"Accuracy={fold_accuracies[i]:.4f}, "
        f"Precision={fold_precisions[i]:.4f}, "
        f"Recall={fold_recalls[i]:.4f}, "
        f"F1={fold_f1s[i]:.4f}"
    )


# ============================================================
# 12. Final Model - Full Training Data
# ============================================================

print("\n")
print("=" * 60)
print("TRAINING FINAL MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Normalize using full training data
# ------------------------------------------------------------

final_mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

final_std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train_final = (
    X_train - final_mean
) / (final_std + 1e-8)

X_test_final = (
    X_test - final_mean
) / (final_std + 1e-8)


# ------------------------------------------------------------
# One-hot
# ------------------------------------------------------------

y_train_final = to_categorical(
    y_train,
    num_classes=6
)


# ------------------------------------------------------------
# Build final model
# ------------------------------------------------------------

tf.keras.backend.clear_session()

final_model = build_model()


# ------------------------------------------------------------
# Train final model
# ------------------------------------------------------------

final_model.fit(
    X_train_final,
    y_train_final,

    epochs=20,

    batch_size=64,

    shuffle=True,

    verbose=1
)


# ============================================================
# 13. Official Test Prediction
# ============================================================

y_prob = final_model.predict(
    X_test_final,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 14. Final Test Results
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n")
print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 15. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 16. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n")
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(cm)


Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)
subjects: (7352,)


FOLD 1
Train shape       : (6012, 128, 9)
Validation shape  : (1340, 128, 9)
Train subjects    : [ 1  3  6  8 11 14 15 16 17 19 21 23 26 27 28 29 30]
Validation subjects: [ 5  7 22 25]
Subject overlap   : []
No subject leakage.
Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.8207 - loss: 0.4896 - val_accuracy: 0.8545 - val_loss: 0.4252
Epoch 2/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9483 - loss: 0.1389 - val_accuracy: 0.9142 - val_loss: 0.2971
Epoch 3/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9561 - loss: 0.1139 - val_accuracy: 0.9157 - val_loss: 0.3396
Epoch 4/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9589 - loss: 0.1042 - val_accuracy: 0.9112 - val_loss: 0.3368
Epoch 5/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9614 - loss: 0.1006 - val_accuracy: 0.9090 - val_loss: 0.4158
Epoch 6/50
94/94 ━━━

In [8]:
import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, ReLU,
    MaxPooling1D, GlobalAveragePooling1D,
    Bidirectional, GRU, Dense, Dropout,
    SpatialDropout1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Dataset
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        X.append(np.loadtxt(file_path))

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y = np.loadtxt(
        f"{DATA_PATH}/{split}/y_{split}.txt"
    ).astype(int) - 1

    # subject 정보
    subject = np.loadtxt(
        f"{DATA_PATH}/{split}/subject_{split}.txt"
    ).astype(int)

    return X, y, subject


X_train, y_train, subjects = load_data("train")
X_test, y_test, test_subjects = load_data("test")

print("Dataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("subjects:", subjects.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ============================================================
# 3. Global normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 4. CNN + BiGRU Model
# ============================================================

def build_model():

    inputs = Input(
        shape=(128, 9)
    )

    # --------------------------------------------------------
    # CNN Block 1
    # --------------------------------------------------------

    x = Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        use_bias=False
    )(inputs)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)

    x = SpatialDropout1D(
        0.15
    )(x)


    # --------------------------------------------------------
    # CNN Block 2
    # --------------------------------------------------------

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)

    x = SpatialDropout1D(
        0.15
    )(x)


    # --------------------------------------------------------
    # BiGRU
    # --------------------------------------------------------

    x = Bidirectional(
        GRU(
            64,
            return_sequences=True,
            dropout=0.15
        )
    )(x)


    # --------------------------------------------------------
    # Global pooling
    # --------------------------------------------------------

    x = GlobalAveragePooling1D()(x)


    # --------------------------------------------------------
    # Dense
    # --------------------------------------------------------

    x = Dense(
        128,
        activation="relu"
    )(x)

    x = BatchNormalization()(x)

    x = Dropout(
        0.35
    )(x)


    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    model = Model(
        inputs=inputs,
        outputs=outputs
    )


    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# ============================================================
# 5. Stratified Group K-Fold
# ============================================================

N_SPLITS = 5

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


fold_results = []


for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X_train,
        y_train,
        groups=subjects
    ),
    start=1
):

    print("\n")
    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)


    X_tr = X_train[train_idx]
    X_val = X_train[val_idx]

    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]

    subject_tr = subjects[train_idx]
    subject_val = subjects[val_idx]


    print("Train shape      :", X_tr.shape)
    print("Validation shape :", X_val.shape)

    print(
        "Train subjects   :",
        np.unique(subject_tr)
    )

    print(
        "Validation subjects:",
        np.unique(subject_val)
    )


    overlap = np.intersect1d(
        np.unique(subject_tr),
        np.unique(subject_val)
    )

    print(
        "Subject overlap  :",
        overlap
    )

    if len(overlap) == 0:
        print("No subject leakage.")
    else:
        raise ValueError(
            "Subject leakage detected!"
        )


    # --------------------------------------------------------
    # One-hot
    # --------------------------------------------------------

    y_tr_onehot = to_categorical(
        y_tr,
        num_classes=6
    )

    y_val_onehot = to_categorical(
        y_val,
        num_classes=6
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    model = build_model()


    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    model.fit(
        X_tr,
        y_tr_onehot,

        validation_data=(
            X_val,
            y_val_onehot
        ),

        epochs=50,

        batch_size=64,

        shuffle=True,

        callbacks=[
            early_stop,
            reduce_lr
        ],

        verbose=1
    )


    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    y_prob = model.predict(
        X_val,
        verbose=0
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )


    fold_results.append([
        accuracy,
        precision,
        recall,
        f1
    ])


    print("\n")
    print("-" * 45)
    print(f"FOLD {fold} RESULTS")
    print("-" * 45)

    print(
        f"Accuracy  : {accuracy:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1-score  : {f1:.4f}"
    )


# ============================================================
# 6. Cross-validation results
# ============================================================

fold_results = np.array(
    fold_results
)

print("\n")
print("=" * 60)
print("STRATIFIED GROUP K-FOLD RESULTS")
print("=" * 60)

print(
    f"Accuracy  : "
    f"{fold_results[:, 0].mean():.4f} "
    f"± {fold_results[:, 0].std():.4f}"
)

print(
    f"Precision : "
    f"{fold_results[:, 1].mean():.4f} "
    f"± {fold_results[:, 1].std():.4f}"
)

print(
    f"Recall    : "
    f"{fold_results[:, 2].mean():.4f} "
    f"± {fold_results[:, 2].std():.4f}"
)

print(
    f"F1-score  : "
    f"{fold_results[:, 3].mean():.4f} "
    f"± {fold_results[:, 3].std():.4f}"
)


print("\n")
print("=" * 60)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 60)

for i, result in enumerate(
    fold_results,
    start=1
):

    print(
        f"Fold {i}: "
        f"Accuracy={result[0]:.4f}, "
        f"Precision={result[1]:.4f}, "
        f"Recall={result[2]:.4f}, "
        f"F1={result[3]:.4f}"
    )


# ============================================================
# 7. Train final model
# ============================================================

print("\n")
print("=" * 60)
print("TRAINING FINAL MODEL")
print("=" * 60)


tf.keras.backend.clear_session()

final_model = build_model()


# CV 결과를 참고해서 안정적인 epoch 수 사용
FINAL_EPOCHS = 15


final_model.fit(
    X_train,
    to_categorical(
        y_train,
        num_classes=6
    ),

    epochs=FINAL_EPOCHS,

    batch_size=64,

    shuffle=True,

    verbose=1
)


# ============================================================
# 8. Final test prediction
# ============================================================

y_prob = final_model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 9. Final test metrics
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n")
print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1-score  : {f1:.4f}"
)


# ============================================================
# 10. Classification report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 11. Confusion matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n")
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

print(cm)

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
subjects: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


FOLD 1
Train shape      : (6012, 128, 9)
Validation shape : (1340, 128, 9)
Train subjects   : [ 1  3  6  8 11 14 15 16 17 19 21 23 26 27 28 29 30]
Validation subjects: [ 5  7 22 25]
Subject overlap  : []
No subject leakage.
Epoch 1/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.8325 - loss: 0.4473 - val_accuracy: 0.9090 - val_loss: 0.4979 - learning_rate: 0.0010
Epoch 2/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9333 - loss: 0.1894 - val_accuracy: 0.9127 - val_loss: 0.2623 - learning_rate: 0.0010
Epoch 3/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.9471 - loss: 0.1414 - val_accuracy: 0.9007 - val_loss: 0.3068 - learning_rate: 0.0010
Epoch 4/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9473 - loss: 0.1359 - val_accuracy: 0.9172 - val_loss: 0.3392 - learning_rate: 0.0010
Epoch 5/50
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - acc

In [9]:
import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    SpatialDropout1D,
    MaxPooling1D,
    Bidirectional,
    GRU,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. SEED
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. GPU 설정
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU:", gpus)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except:
            pass

else:
    print("GPU not found. Using CPU.")


# ============================================================
# 3. 데이터 확인
# ============================================================
#
# 이미 X_train, y_train, subjects, X_test, y_test가
# 메모리에 있다면 이 부분은 그대로 사용하면 됩니다.
#
# 예상 shape:
#
# X_train  : (7352, 128, 9)
# y_train  : (7352,)
# subjects : (7352,)
# X_test   : (2947, 128, 9)
# y_test   : (2947,)
#
# ============================================================

print("=" * 60)
print("ORIGINAL DATA")
print("=" * 60)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("subjects:", subjects.shape)
print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)


# ============================================================
# 4. LABEL 확인
# ============================================================

# label이 1~6이면 0~5로 변환
if np.min(y_train) == 1:
    y_train = y_train.astype(np.int32) - 1
    y_test = y_test.astype(np.int32) - 1

else:
    y_train = y_train.astype(np.int32)
    y_test = y_test.astype(np.int32)


NUM_CLASSES = len(np.unique(y_train))

print("\nNumber of classes:", NUM_CLASSES)
print("Classes:", np.unique(y_train))


# ============================================================
# 5. SENSOR FEATURE ENGINEERING
# ============================================================
#
# 기존 9축
#
# 0,1,2 = Accelerometer
# 3,4,5 = Gyroscope
# 6,7,8 = 기타 센서
#
# 여기에
#
# acc magnitude
# gyro magnitude
#
# 2개 추가
#
# 9 -> 11 channels
# ============================================================

def add_sensor_features(X):

    X = X.astype(np.float32)

    acc = X[:, :, 0:3]
    gyro = X[:, :, 3:6]

    acc_mag = np.sqrt(
        np.sum(acc ** 2, axis=-1, keepdims=True)
    )

    gyro_mag = np.sqrt(
        np.sum(gyro ** 2, axis=-1, keepdims=True)
    )

    X_new = np.concatenate(
        [
            X,
            acc_mag,
            gyro_mag
        ],
        axis=-1
    )

    return X_new


print("\nAdding sensor magnitude features...")

X_train_feat = add_sensor_features(X_train)
X_test_feat = add_sensor_features(X_test)

print("New X_train shape:", X_train_feat.shape)
print("New X_test shape :", X_test_feat.shape)


# ============================================================
# 6. AUGMENTATION
# ============================================================

def augment_sensor_data(
    X,
    noise_std=0.01,
    scale_range=(0.95, 1.05),
    time_shift=3
):

    X_aug = X.copy()

    # --------------------------------------------------------
    # Gaussian noise
    # --------------------------------------------------------

    noise = np.random.normal(
        loc=0.0,
        scale=noise_std,
        size=X_aug.shape
    ).astype(np.float32)

    X_aug += noise


    # --------------------------------------------------------
    # Amplitude scaling
    # --------------------------------------------------------

    scale = np.random.uniform(
        scale_range[0],
        scale_range[1],
        size=(X.shape[0], 1, X.shape[2])
    ).astype(np.float32)

    X_aug *= scale


    # --------------------------------------------------------
    # Small temporal shift
    # --------------------------------------------------------

    if time_shift > 0:

        shifts = np.random.randint(
            -time_shift,
            time_shift + 1,
            size=X.shape[0]
        )

        for i, shift in enumerate(shifts):

            if shift > 0:
                X_aug[i] = np.concatenate(
                    [
                        X_aug[i, shift:],
                        np.repeat(
                            X_aug[i, -1:],
                            shift,
                            axis=0
                        )
                    ],
                    axis=0
                )

            elif shift < 0:

                s = abs(shift)

                X_aug[i] = np.concatenate(
                    [
                        np.repeat(
                            X_aug[i, :1],
                            s,
                            axis=0
                        ),
                        X_aug[i, :-s]
                    ],
                    axis=0
                )

    return X_aug.astype(np.float32)


# ============================================================
# 7. NORMALIZATION
# ============================================================

def normalize_train_val(
    X_train,
    X_val
):

    n_channels = X_train.shape[-1]

    scaler = StandardScaler()

    X_train_2d = X_train.reshape(
        -1,
        n_channels
    )

    X_val_2d = X_val.reshape(
        -1,
        n_channels
    )

    scaler.fit(X_train_2d)

    X_train_scaled = scaler.transform(
        X_train_2d
    ).reshape(X_train.shape)

    X_val_scaled = scaler.transform(
        X_val_2d
    ).reshape(X_val.shape)

    return (
        X_train_scaled.astype(np.float32),
        X_val_scaled.astype(np.float32),
        scaler
    )


def normalize_test(
    X_test,
    scaler
):

    n_channels = X_test.shape[-1]

    X_test_2d = X_test.reshape(
        -1,
        n_channels
    )

    X_test_scaled = scaler.transform(
        X_test_2d
    ).reshape(X_test.shape)

    return X_test_scaled.astype(np.float32)


# ============================================================
# 8. MODEL
# ============================================================

def build_model(
    input_shape,
    num_classes
):

    inputs = Input(
        shape=input_shape,
        name="sensor_input"
    )


    # ========================================================
    # CNN BLOCK 1
    # ========================================================

    x = Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(inputs)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.15
    )(x)


    # ========================================================
    # CNN BLOCK 2
    # ========================================================

    x = Conv1D(
        filters=128,
        kernel_size=5,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.15
    )(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)


    # ========================================================
    # CNN BLOCK 3
    # ========================================================

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.10
    )(x)


    # ========================================================
    # BiGRU
    # ========================================================

    x = Bidirectional(
        GRU(
            64,
            return_sequences=True,
            dropout=0.15,
            recurrent_dropout=0.0
        )
    )(x)


    # ========================================================
    # SELF ATTENTION
    # ========================================================

    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.10
    )(
        x,
        x
    )

    x = LayerNormalization()(
        x + attention
    )


    # ========================================================
    # GLOBAL POOLING
    # ========================================================

    x = GlobalAveragePooling1D()(x)


    # ========================================================
    # CLASSIFIER
    # ========================================================

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(x)

    x = Dropout(
        0.35
    )(x)


    outputs = Dense(
        num_classes,
        activation="softmax",
        name="output"
    )(x)


    model = Model(
        inputs,
        outputs
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=3e-4
    )


    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


    return model


# ============================================================
# 9. CALLBACKS
# ============================================================

def get_callbacks():

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )

    return [
        early_stop,
        reduce_lr
    ]


# ============================================================
# 10. STRATIFIED GROUP K-FOLD
# ============================================================

N_SPLITS = 5

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


fold_results = []


# ============================================================
# 11. CROSS VALIDATION
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X_train_feat,
        y_train,
        groups=subjects
    ),
    start=1
):

    print("\n")
    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)


    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    X_tr = X_train_feat[train_idx]
    X_val = X_train_feat[val_idx]

    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]

    subject_tr = subjects[train_idx]
    subject_val = subjects[val_idx]


    print("Train shape      :", X_tr.shape)
    print("Validation shape :", X_val.shape)

    print(
        "Train subjects   :",
        np.unique(subject_tr)
    )

    print(
        "Validation subjects:",
        np.unique(subject_val)
    )


    overlap = np.intersect1d(
        np.unique(subject_tr),
        np.unique(subject_val)
    )

    print(
        "Subject overlap  :",
        overlap
    )

    if len(overlap) == 0:
        print("No subject leakage.")
    else:
        raise ValueError(
            "Subject leakage detected!"
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    X_tr, X_val, scaler = normalize_train_val(
        X_tr,
        X_val
    )


    # --------------------------------------------------------
    # Augmentation
    # --------------------------------------------------------

    X_aug = augment_sensor_data(
        X_tr,
        noise_std=0.01,
        scale_range=(0.97, 1.03),
        time_shift=2
    )

    X_tr_final = np.concatenate(
        [
            X_tr,
            X_aug
        ],
        axis=0
    )

    y_tr_final = np.concatenate(
        [
            y_tr,
            y_tr
        ],
        axis=0
    )


    print(
        "Augmented train shape:",
        X_tr_final.shape
    )


    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    model = build_model(
        input_shape=X_tr.shape[1:],
        num_classes=NUM_CLASSES
    )


    if fold == 1:
        model.summary()


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(
        X_tr_final,
        y_tr_final,

        validation_data=(
            X_val,
            y_val
        ),

        epochs=50,

        batch_size=64,

        shuffle=True,

        callbacks=get_callbacks(),

        verbose=1
    )


    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    y_prob = model.predict(
        X_val,
        batch_size=256,
        verbose=0
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    acc = accuracy_score(
        y_val,
        y_pred
    )

    precision = precision_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )


    fold_results.append(
        [
            acc,
            precision,
            recall,
            f1
        ]
    )


    print("\n")
    print("-" * 45)
    print(f"FOLD {fold} RESULTS")
    print("-" * 45)

    print(
        f"Accuracy  : {acc:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1-score  : {f1:.4f}"
    )


    # --------------------------------------------------------
    # Fold confusion matrix
    # --------------------------------------------------------

    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            y_val,
            y_pred
        )
    )


# ============================================================
# 12. CV RESULT
# ============================================================

fold_results = np.array(
    fold_results
)

print("\n")
print("=" * 60)
print("STRATIFIED GROUP K-FOLD RESULTS")
print("=" * 60)

print(
    f"Accuracy  : "
    f"{fold_results[:,0].mean():.4f} "
    f"± "
    f"{fold_results[:,0].std():.4f}"
)

print(
    f"Precision : "
    f"{fold_results[:,1].mean():.4f} "
    f"± "
    f"{fold_results[:,1].std():.4f}"
)

print(
    f"Recall    : "
    f"{fold_results[:,2].mean():.4f} "
    f"± "
    f"{fold_results[:,2].std():.4f}"
)

print(
    f"F1-score  : "
    f"{fold_results[:,3].mean():.4f} "
    f"± "
    f"{fold_results[:,3].std():.4f}"
)


print("\n")
print("=" * 60)
print("INDIVIDUAL FOLD RESULTS")
print("=" * 60)

for i, result in enumerate(
    fold_results,
    start=1
):

    print(
        f"Fold {i}: "
        f"Accuracy={result[0]:.4f}, "
        f"Precision={result[1]:.4f}, "
        f"Recall={result[2]:.4f}, "
        f"F1={result[3]:.4f}"
    )


# ============================================================
# 13. FINAL MODEL
# ============================================================

print("\n")
print("=" * 60)
print("TRAINING FINAL MODEL")
print("=" * 60)


# ------------------------------------------------------------
# Normalize full training set
# ------------------------------------------------------------

n_channels = X_train_feat.shape[-1]

final_scaler = StandardScaler()

X_train_2d = X_train_feat.reshape(
    -1,
    n_channels
)

X_test_2d = X_test_feat.reshape(
    -1,
    n_channels
)

final_scaler.fit(
    X_train_2d
)

X_train_scaled = final_scaler.transform(
    X_train_2d
).reshape(
    X_train_feat.shape
).astype(np.float32)

X_test_scaled = final_scaler.transform(
    X_test_2d
).reshape(
    X_test_feat.shape
).astype(np.float32
)


# ------------------------------------------------------------
# Final augmentation
# ------------------------------------------------------------

X_aug = augment_sensor_data(
    X_train_scaled,
    noise_std=0.01,
    scale_range=(0.97, 1.03),
    time_shift=2
)

X_train_final = np.concatenate(
    [
        X_train_scaled,
        X_aug
    ],
    axis=0
)

y_train_final = np.concatenate(
    [
        y_train,
        y_train
    ],
    axis=0
)


print(
    "Final train shape:",
    X_train_final.shape
)

print(
    "Test shape:",
    X_test_scaled.shape
)


# ------------------------------------------------------------
# Build final model
# ------------------------------------------------------------

final_model = build_model(
    input_shape=X_train_scaled.shape[1:],
    num_classes=NUM_CLASSES
)


# ------------------------------------------------------------
# Final training
# ------------------------------------------------------------

final_model.fit(
    X_train_final,
    y_train_final,

    epochs=20,

    batch_size=64,

    shuffle=True,

    verbose=1
)


# ============================================================
# 14. TEST
# ============================================================

print("\n")
print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)


test_prob = final_model.predict(
    X_test_scaled,
    batch_size=256,
    verbose=1
)

test_pred = np.argmax(
    test_prob,
    axis=1
)


# ============================================================
# 15. METRICS
# ============================================================

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

test_precision = precision_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)


print(
    f"Accuracy  : {test_accuracy:.4f}"
)

print(
    f"Precision : {test_precision:.4f}"
)

print(
    f"Recall    : {test_recall:.4f}"
)

print(
    f"F1-score  : {test_f1:.4f}"
)


# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        test_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 17. CONFUSION MATRIX
# ============================================================

print("\n")
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(
    y_test,
    test_pred
)

print(cm)


# ============================================================
# 18. CLASS별 정확도
# ============================================================

print("\n")
print("=" * 60)
print("CLASS ACCURACY")
print("=" * 60)

for i, name in enumerate(class_names):

    total = np.sum(
        cm[i]
    )

    correct = cm[i, i]

    class_acc = (
        correct / total
        if total > 0
        else 0
    )

    print(
        f"{name:20s}: "
        f"{class_acc:.4f}"
    )

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
ORIGINAL DATA
X_train : (7352, 128, 9)
y_train : (7352,)
subjects: (7352,)
X_test  : (2947, 128, 9)
y_test  : (2947,)

Number of classes: 6
Classes: [0 1 2 3 4 5]

Adding sensor magnitude features...
New X_train shape: (7352, 128, 11)
New X_test shape : (2947, 128, 11)


FOLD 1
Train shape      : (6012, 128, 11)
Validation shape : (1340, 128, 11)
Train subjects   : [ 1  3  6  8 11 14 15 16 17 19 21 23 26 27 28 29 30]
Validation subjects: [ 5  7 22 25]
Subject overlap  : []
No subject leakage.
Augmented train shape: (12024, 128, 11)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_input        │ (None, 128, 11)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 128, 64)   │      3,584 │ sensor_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_2 │ (None, 128, 64)   │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 128, 128)  │     41,088 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_3 │ (None, 128, 128)  │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 64, 128)   │          0 │ spatial_dropout1… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 64, 128)   │     49,280 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 128)   │        512 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_4 │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64, 128)   │     74,496 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 64, 128)   │     66,048 │ bidirectional_1[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 128)   │          0 │ bidirectional_1[… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 64, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 244,934 (956.77 KB)

 Trainable params: 244,166 (953.77 KB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.8452 - loss: 0.4766 - val_accuracy: 0.7955 - val_loss: 0.5483 - learning_rate: 3.0000e-04
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9310 - loss: 0.2198 - val_accuracy: 0.9127 - val_loss: 0.3008 - learning_rate: 3.0000e-04
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9423 - loss: 0.1894 - val_accuracy: 0.9149 - val_loss: 0.2533 - learning_rate: 3.0000e-04
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.9481 - loss: 0.1711 - val_accuracy: 0.9216 - val_loss: 0.3632 - learning_rate: 3.0000e-04
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9508 - loss: 0.1617 - val_accuracy: 0.9179 - val_loss: 0.4154 - learning_rate: 3.0000e-04
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9511 - loss: 0.1498
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accurac



---------------------------------------------
FOLD 2 RESULTS
---------------------------------------------
Accuracy  : 0.9666
Precision : 0.9705
Recall    : 0.9666
F1-score  : 0.9664

Confusion Matrix:
[[250   3   0   0   0   0]
 [  0 191   0   0   0   0]
 [  0   0 179   0   0   0]
 [  0   0   0 233  44   0]
 [  0   1   0   0 293   0]
 [  0   2   0   0   0 301]]


FOLD 3
Train shape      : (5624, 128, 11)
Validation shape : (1728, 128, 11)
Train subjects   : [ 1  3  5  6  7 14 15 16 17 19 21 22 25 26 28 29]
Validation subjects: [ 8 11 23 27 30]
Subject overlap  : []
No subject leakage.
Augmented train shape: (11248, 128, 11)
Epoch 1/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.8151 - loss: 0.5392 - val_accuracy: 0.7506 - val_loss: 0.5296 - learning_rate: 3.0000e-04
Epoch 2/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.9238 - loss: 0.2440 - val_accuracy: 0.9549 - val_loss: 0.1307 - learning_rate: 3.0000e-04
Epoch 3/50
176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/st



---------------------------------------------
FOLD 3 RESULTS
---------------------------------------------
Accuracy  : 0.9722
Precision : 0.9744
Recall    : 0.9722
F1-score  : 0.9721

Confusion Matrix:
[[288   0   0   0   0   0]
 [  0 262   0   0   0   0]
 [  0   0 244   0   0   0]
 [  0   0   0 293   6   0]
 [  0   0   0  42 266   0]
 [  0   0   0   0   0 327]]


FOLD 4
Train shape      : (5943, 128, 11)
Validation shape : (1409, 128, 11)
Train subjects   : [ 1  3  5  6  7  8 11 17 19 21 22 23 25 27 28 29 30]
Validation subjects: [14 15 16 26]
Subject overlap  : []
No subject leakage.
Augmented train shape: (11886, 128, 11)
Epoch 1/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 11s 31ms/step - accuracy: 0.8544 - loss: 0.4482 - val_accuracy: 0.8119 - val_loss: 0.6509 - learning_rate: 3.0000e-04
Epoch 2/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - accuracy: 0.9409 - loss: 0.1938 - val_accuracy: 0.8275 - val_loss: 0.5875 - learning_rate: 3.0000e-04
Epoch 3/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/st

In [10]:
import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (
    Conv1D,
    BatchNormalization,
    SpatialDropout1D,
    MaxPooling1D,
    Bidirectional,
    GRU,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from tensorflow.keras.regularizers import l2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. SEED
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. GPU 설정
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:

    print("GPU:", gpus)

    for gpu in gpus:

        try:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        except Exception:
            pass

else:

    print("GPU not found. Using CPU.")


# ============================================================
# 3. DATA 확인
# ============================================================

print("=" * 60)
print("ORIGINAL DATA")
print("=" * 60)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("subjects:", subjects.shape)
print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)


# ============================================================
# 4. LABEL 확인
# ============================================================

# label이 1~6이면 0~5로 변환

if np.min(y_train) == 1:

    y_train = (
        y_train
        .astype(np.int32)
        - 1
    )

    y_test = (
        y_test
        .astype(np.int32)
        - 1
    )

else:

    y_train = y_train.astype(
        np.int32
    )

    y_test = y_test.astype(
        np.int32
    )


NUM_CLASSES = len(
    np.unique(y_train)
)

print(
    "\nNumber of classes:",
    NUM_CLASSES
)

print(
    "Classes:",
    np.unique(y_train)
)


# ============================================================
# 5. SENSOR FEATURE ENGINEERING
# ============================================================

def add_sensor_features(X):

    X = X.astype(
        np.float32
    )

    # Accelerometer
    acc = X[:, :, 0:3]

    # Gyroscope
    gyro = X[:, :, 3:6]

    # Accelerometer magnitude
    acc_mag = np.sqrt(
        np.sum(
            acc ** 2,
            axis=-1,
            keepdims=True
        )
    )

    # Gyroscope magnitude
    gyro_mag = np.sqrt(
        np.sum(
            gyro ** 2,
            axis=-1,
            keepdims=True
        )
    )

    X_new = np.concatenate(
        [
            X,
            acc_mag,
            gyro_mag
        ],
        axis=-1
    )

    return X_new


print(
    "\nAdding sensor magnitude features..."
)

X_train_feat = add_sensor_features(
    X_train
)

X_test_feat = add_sensor_features(
    X_test
)

print(
    "New X_train shape:",
    X_train_feat.shape
)

print(
    "New X_test shape :",
    X_test_feat.shape
)


# ============================================================
# 6. AUGMENTATION
# ============================================================

def augment_sensor_data(
    X,
    noise_std=0.01,
    scale_range=(0.95, 1.05),
    time_shift=3
):

    X_aug = X.copy()


    # --------------------------------------------------------
    # Gaussian noise
    # --------------------------------------------------------

    noise = np.random.normal(
        loc=0.0,
        scale=noise_std,
        size=X_aug.shape
    ).astype(
        np.float32
    )

    X_aug += noise


    # --------------------------------------------------------
    # Amplitude scaling
    # --------------------------------------------------------

    scale = np.random.uniform(
        scale_range[0],
        scale_range[1],
        size=(
            X.shape[0],
            1,
            X.shape[2]
        )
    ).astype(
        np.float32
    )

    X_aug *= scale


    # --------------------------------------------------------
    # Small temporal shift
    # --------------------------------------------------------

    if time_shift > 0:

        shifts = np.random.randint(
            -time_shift,
            time_shift + 1,
            size=X.shape[0]
        )

        for i, shift in enumerate(shifts):

            if shift > 0:

                X_aug[i] = np.concatenate(
                    [
                        X_aug[i, shift:],

                        np.repeat(
                            X_aug[i, -1:],
                            shift,
                            axis=0
                        )
                    ],
                    axis=0
                )

            elif shift < 0:

                s = abs(shift)

                X_aug[i] = np.concatenate(
                    [
                        np.repeat(
                            X_aug[i, :1],
                            s,
                            axis=0
                        ),

                        X_aug[i, :-s]
                    ],
                    axis=0
                )

    return X_aug.astype(
        np.float32
    )


# ============================================================
# 7. NORMALIZATION
# ============================================================

def normalize_train_val(
    X_train,
    X_val
):

    n_channels = X_train.shape[-1]

    scaler = StandardScaler()


    X_train_2d = X_train.reshape(
        -1,
        n_channels
    )

    X_val_2d = X_val.reshape(
        -1,
        n_channels
    )


    # Train에만 fit
    scaler.fit(
        X_train_2d
    )


    X_train_scaled = scaler.transform(
        X_train_2d
    ).reshape(
        X_train.shape
    )


    X_val_scaled = scaler.transform(
        X_val_2d
    ).reshape(
        X_val.shape
    )


    return (
        X_train_scaled.astype(
            np.float32
        ),

        X_val_scaled.astype(
            np.float32
        ),

        scaler
    )


def normalize_test(
    X_test,
    scaler
):

    n_channels = X_test.shape[-1]

    X_test_2d = X_test.reshape(
        -1,
        n_channels
    )

    X_test_scaled = scaler.transform(
        X_test_2d
    ).reshape(
        X_test.shape
    )

    return X_test_scaled.astype(
        np.float32
    )


# ============================================================
# 8. MODEL
# ============================================================

def build_model(
    input_shape,
    num_classes
):

    inputs = Input(
        shape=input_shape,
        name="sensor_input"
    )


    # ========================================================
    # CNN BLOCK 1
    # ========================================================

    x = Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(inputs)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.15
    )(x)


    # ========================================================
    # CNN BLOCK 2
    # ========================================================

    x = Conv1D(
        filters=128,
        kernel_size=5,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.15
    )(x)

    x = MaxPooling1D(
        pool_size=2
    )(x)


    # ========================================================
    # CNN BLOCK 3
    # ========================================================

    x = Conv1D(
        filters=128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(x)

    x = SpatialDropout1D(
        0.10
    )(x)


    # ========================================================
    # BiGRU
    # ========================================================

    x = Bidirectional(
        GRU(
            64,
            return_sequences=True,
            dropout=0.15,
            recurrent_dropout=0.0
        )
    )(x)


    # ========================================================
    # SELF ATTENTION
    # ========================================================

    attention = MultiHeadAttention(
        num_heads=4,
        key_dim=32,
        dropout=0.10
    )(
        x,
        x
    )

    x = LayerNormalization()(
        x + attention
    )


    # ========================================================
    # GLOBAL POOLING
    # ========================================================

    x = GlobalAveragePooling1D()(
        x
    )


    # ========================================================
    # CLASSIFIER
    # ========================================================

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(1e-4)
    )(x)

    x = BatchNormalization()(
        x
    )

    x = Dropout(
        0.35
    )(x)


    # ========================================================
    # OUTPUT
    # ========================================================

    outputs = Dense(
        num_classes,
        activation="softmax",
        name="output"
    )(x)


    model = Model(
        inputs,
        outputs
    )


    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=3e-4
    )


    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


    return model


# ============================================================
# 9. CALLBACKS
# ============================================================

def get_callbacks():

    early_stop = EarlyStopping(
        monitor="val_loss",

        # 최대 7 epoch 동안 개선이 없으면 종료
        patience=7,

        restore_best_weights=True,

        verbose=1
    )


    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",

        factor=0.5,

        patience=3,

        min_lr=1e-6,

        verbose=1
    )


    return [
        early_stop,
        reduce_lr
    ]


# ============================================================
# 10. STRATIFIED GROUP K-FOLD
# ============================================================

N_SPLITS = 5

cv = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)


fold_results = []

# ★ 추가
# 각 fold에서 best epoch 저장
best_epochs = []


# ============================================================
# 11. CROSS VALIDATION
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X_train_feat,
        y_train,
        groups=subjects
    ),
    start=1
):

    print("\n")
    print("=" * 60)
    print(f"FOLD {fold}")
    print("=" * 60)


    # --------------------------------------------------------
    # Split
    # --------------------------------------------------------

    X_tr = X_train_feat[
        train_idx
    ]

    X_val = X_train_feat[
        val_idx
    ]

    y_tr = y_train[
        train_idx
    ]

    y_val = y_train[
        val_idx
    ]

    subject_tr = subjects[
        train_idx
    ]

    subject_val = subjects[
        val_idx
    ]


    print(
        "Train shape      :",
        X_tr.shape
    )

    print(
        "Validation shape :",
        X_val.shape
    )


    print(
        "Train subjects   :",
        np.unique(subject_tr)
    )

    print(
        "Validation subjects:",
        np.unique(subject_val)
    )


    # --------------------------------------------------------
    # Subject leakage 확인
    # --------------------------------------------------------

    overlap = np.intersect1d(
        np.unique(subject_tr),
        np.unique(subject_val)
    )


    print(
        "Subject overlap  :",
        overlap
    )


    if len(overlap) == 0:

        print(
            "No subject leakage."
        )

    else:

        raise ValueError(
            "Subject leakage detected!"
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    X_tr, X_val, scaler = normalize_train_val(
        X_tr,
        X_val
    )


    # --------------------------------------------------------
    # Augmentation
    # --------------------------------------------------------

    X_aug = augment_sensor_data(
        X_tr,

        noise_std=0.01,

        scale_range=(
            0.97,
            1.03
        ),

        time_shift=2
    )


    X_tr_final = np.concatenate(
        [
            X_tr,
            X_aug
        ],
        axis=0
    )


    y_tr_final = np.concatenate(
        [
            y_tr,
            y_tr
        ],
        axis=0
    )


    print(
        "Augmented train shape:",
        X_tr_final.shape
    )


    # --------------------------------------------------------
    # Build model
    # --------------------------------------------------------

    model = build_model(
        input_shape=X_tr.shape[1:],
        num_classes=NUM_CLASSES
    )


    if fold == 1:

        model.summary()


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(

        X_tr_final,

        y_tr_final,

        validation_data=(
            X_val,
            y_val
        ),

        # ★ CV에서는 최대 50 epoch
        # EarlyStopping이 실제 종료 시점을 결정
        epochs=50,

        batch_size=64,

        shuffle=True,

        callbacks=get_callbacks(),

        verbose=1
    )


    # ========================================================
    # ★ BEST EPOCH 계산
    # ========================================================

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        )
        + 1
    )


    best_epochs.append(
        best_epoch
    )


    print(
        f"\nFold {fold} best epoch: "
        f"{best_epoch}"
    )


    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    y_prob = model.predict(
        X_val,
        batch_size=256,
        verbose=0
    )


    y_pred = np.argmax(
        y_prob,
        axis=1
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    acc = accuracy_score(
        y_val,
        y_pred
    )


    precision = precision_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )


    recall = recall_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )


    f1 = f1_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )


    fold_results.append(
        [
            acc,
            precision,
            recall,
            f1
        ]
    )


    # --------------------------------------------------------
    # Fold 결과
    # --------------------------------------------------------

    print("\n")
    print("-" * 45)
    print(
        f"FOLD {fold} RESULTS"
    )
    print("-" * 45)


    print(
        f"Best Epoch: {best_epoch}"
    )

    print(
        f"Accuracy  : {acc:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1-score  : {f1:.4f}"
    )


    # --------------------------------------------------------
    # Fold confusion matrix
    # --------------------------------------------------------

    print(
        "\nConfusion Matrix:"
    )

    print(
        confusion_matrix(
            y_val,
            y_pred
        )
    )


# ============================================================
# 12. CV RESULT
# ============================================================

fold_results = np.array(
    fold_results
)


print("\n")
print("=" * 60)
print(
    "STRATIFIED GROUP K-FOLD RESULTS"
)
print("=" * 60)


print(
    f"Accuracy  : "
    f"{fold_results[:,0].mean():.4f} "
    f"± "
    f"{fold_results[:,0].std():.4f}"
)


print(
    f"Precision : "
    f"{fold_results[:,1].mean():.4f} "
    f"± "
    f"{fold_results[:,1].std():.4f}"
)


print(
    f"Recall    : "
    f"{fold_results[:,2].mean():.4f} "
    f"± "
    f"{fold_results[:,2].std():.4f}"
)


print(
    f"F1-score  : "
    f"{fold_results[:,3].mean():.4f} "
    f"± "
    f"{fold_results[:,3].std():.4f}"
)


# ============================================================
# 13. BEST EPOCH RESULT
# ============================================================

print("\n")
print("=" * 60)
print(
    "BEST EPOCH ANALYSIS"
)
print("=" * 60)


for i, epoch in enumerate(
    best_epochs,
    start=1
):

    print(
        f"Fold {i}: "
        f"Best Epoch = {epoch}"
    )


# ------------------------------------------------------------
# Median best epoch
# ------------------------------------------------------------

final_epochs = int(
    np.median(
        best_epochs
    )
)


# ------------------------------------------------------------
# Mean best epoch
# ------------------------------------------------------------

mean_best_epoch = int(
    round(
        np.mean(
            best_epochs
        )
    )
)


print(
    "\nBest epochs:",
    best_epochs
)

print(
    "Mean best epoch:",
    mean_best_epoch
)

print(
    "Median best epoch:",
    final_epochs
)


# ============================================================
# 14. INDIVIDUAL FOLD RESULTS
# ============================================================

print("\n")
print("=" * 60)
print(
    "INDIVIDUAL FOLD RESULTS"
)
print("=" * 60)


for i, result in enumerate(
    fold_results,
    start=1
):

    print(
        f"Fold {i}: "
        f"Accuracy={result[0]:.4f}, "
        f"Precision={result[1]:.4f}, "
        f"Recall={result[2]:.4f}, "
        f"F1={result[3]:.4f}, "
        f"BestEpoch={best_epochs[i-1]}"
    )


# ============================================================
# 15. FINAL MODEL
# ============================================================

print("\n")
print("=" * 60)
print(
    "TRAINING FINAL MODEL"
)
print("=" * 60)


print(
    f"Final training epochs: "
    f"{final_epochs}"
)


# ============================================================
# Normalize full training set
# ============================================================

n_channels = X_train_feat.shape[-1]

final_scaler = StandardScaler()


X_train_2d = X_train_feat.reshape(
    -1,
    n_channels
)


X_test_2d = X_test_feat.reshape(
    -1,
    n_channels
)


# ------------------------------------------------------------
# IMPORTANT
# scaler는 training data에만 fit
# ------------------------------------------------------------

final_scaler.fit(
    X_train_2d
)


X_train_scaled = final_scaler.transform(
    X_train_2d
).reshape(
    X_train_feat.shape
).astype(
    np.float32
)


X_test_scaled = final_scaler.transform(
    X_test_2d
).reshape(
    X_test_feat.shape
).astype(
    np.float32
)


# ============================================================
# 16. FINAL AUGMENTATION
# ============================================================

X_aug = augment_sensor_data(
    X_train_scaled,

    noise_std=0.01,

    scale_range=(
        0.97,
        1.03
    ),

    time_shift=2
)


X_train_final = np.concatenate(
    [
        X_train_scaled,
        X_aug
    ],
    axis=0
)


y_train_final = np.concatenate(
    [
        y_train,
        y_train
    ],
    axis=0
)


print(
    "Final train shape:",
    X_train_final.shape
)


print(
    "Test shape:",
    X_test_scaled.shape
)


# ============================================================
# 17. BUILD FINAL MODEL
# ============================================================

final_model = build_model(
    input_shape=X_train_scaled.shape[1:],
    num_classes=NUM_CLASSES
)


# ============================================================
# 18. FINAL TRAINING
# ============================================================
#
# CV에서 얻은 median best epoch를 사용
#
# 여기서는 validation set이 없으므로
# EarlyStopping을 사용하지 않음.
#
# ============================================================

final_model.fit(

    X_train_final,

    y_train_final,

    epochs=final_epochs,

    batch_size=64,

    shuffle=True,

    verbose=1
)


# ============================================================
# 19. TEST
# ============================================================

print("\n")
print("=" * 60)
print(
    "FINAL TEST RESULTS"
)
print("=" * 60)


test_prob = final_model.predict(
    X_test_scaled,
    batch_size=256,
    verbose=1
)


test_pred = np.argmax(
    test_prob,
    axis=1
)


# ============================================================
# 20. METRICS
# ============================================================

test_accuracy = accuracy_score(
    y_test,
    test_pred
)


test_precision = precision_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)


test_recall = recall_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)


test_f1 = f1_score(
    y_test,
    test_pred,
    average="weighted",
    zero_division=0
)


print(
    f"Accuracy  : "
    f"{test_accuracy:.4f}"
)


print(
    f"Precision : "
    f"{test_precision:.4f}"
)


print(
    f"Recall    : "
    f"{test_recall:.4f}"
)


print(
    f"F1-score  : "
    f"{test_f1:.4f}"
)


# ============================================================
# 21. CLASSIFICATION REPORT
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n")
print("=" * 60)
print(
    "CLASSIFICATION REPORT"
)
print("=" * 60)


print(
    classification_report(
        y_test,
        test_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 22. CONFUSION MATRIX
# ============================================================

print("\n")
print("=" * 60)
print(
    "CONFUSION MATRIX"
)
print("=" * 60)


cm = confusion_matrix(
    y_test,
    test_pred
)


print(cm)


# ============================================================
# 23. CLASS ACCURACY
# ============================================================

print("\n")
print("=" * 60)
print(
    "CLASS ACCURACY"
)
print("=" * 60)


for i, name in enumerate(
    class_names
):

    total = np.sum(
        cm[i]
    )

    correct = cm[i, i]


    class_acc = (
        correct / total
        if total > 0
        else 0
    )


    print(
        f"{name:20s}: "
        f"{class_acc:.4f}"
    )


# ============================================================
# 24. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print(
    "FINAL SUMMARY"
)
print("=" * 60)


print(
    "CV Best Epochs:",
    best_epochs
)


print(
    "Final Epochs:",
    final_epochs
)


print(
    f"CV Accuracy: "
    f"{fold_results[:,0].mean():.4f} "
    f"± "
    f"{fold_results[:,0].std():.4f}"
)


print(
    f"CV F1-score: "
    f"{fold_results[:,3].mean():.4f} "
    f"± "
    f"{fold_results[:,3].std():.4f}"
)


print(
    f"Test Accuracy: "
    f"{test_accuracy:.4f}"
)


print(
    f"Test F1-score: "
    f"{test_f1:.4f}"
)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
ORIGINAL DATA
X_train : (7352, 128, 9)
y_train : (7352,)
subjects: (7352,)
X_test  : (2947, 128, 9)
y_test  : (2947,)

Number of classes: 6
Classes: [0 1 2 3 4 5]

Adding sensor magnitude features...
New X_train shape: (7352, 128, 11)
New X_test shape : (2947, 128, 11)


FOLD 1
Train shape      : (6012, 128, 11)
Validation shape : (1340, 128, 11)
Train subjects   : [ 1  3  6  8 11 14 15 16 17 19 21 23 26 27 28 29 30]
Validation subjects: [ 5  7 22 25]
Subject overlap  : []
No subject leakage.
Augmented train shape: (12024, 128, 11)


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_input        │ (None, 128, 11)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_22 (Conv1D)  │ (None, 128, 64)   │      3,584 │ sensor_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 64)   │        256 │ conv1d_22[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 128, 64)   │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_23 (Conv1D)  │ (None, 128, 128)  │     41,088 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_23[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 128, 128)  │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_8     │ (None, 64, 128)   │          0 │ spatial_dropout1… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_24 (Conv1D)  │ (None, 64, 128)   │     49,280 │ max_pooling1d_8[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 128)   │        512 │ conv1d_24[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_… │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_7     │ (None, 64, 128)   │     74,496 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 64, 128)   │     66,048 │ bidirectional_7[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_7[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 64, 128)   │          0 │ bidirectional_7[… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 64, 128)   │        256 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 244,934 (956.77 KB)

 Trainable params: 244,166 (953.77 KB)

 Non-trainable params: 768 (3.00 KB)

Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - accuracy: 0.8446 - loss: 0.4768 - val_accuracy: 0.7993 - val_loss: 0.5415 - learning_rate: 3.0000e-04
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.9309 - loss: 0.2198 - val_accuracy: 0.9142 - val_loss: 0.3124 - learning_rate: 3.0000e-04
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.9420 - loss: 0.1899 - val_accuracy: 0.9254 - val_loss: 0.2515 - learning_rate: 3.0000e-04
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9481 - loss: 0.1714 - val_accuracy: 0.9209 - val_loss: 0.3744 - learning_rate: 3.0000e-04
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.9508 - loss: 0.1628 - val_accuracy: 0.9231 - val_loss: 0.4030 - learning_rate: 3.0000e-04
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9511 - loss: 0.1510
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0001500000071246177.
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accurac